# W5 Results Evaluation (molecule, facts_50k + paper)

Evaluate baselines against brute-force ground truth for W5 (outer jaccard k-NN on `facts_50k.mol_ecfp` with inner L2 partner lookup on `paper.abstract_emb` within τ).

**Recall**: for each query, count baseline results with `score <= worst_GT_score`. `recall = hits / K`. We only compare scores, not (fact_id, pmid) — user instruction.

In [1]:
import json
from pathlib import Path

RESULTS_DIR = Path(".")
GT_FILE = "w5_queries_100_gt.json"

def load_results(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def gt_answer_to_scores(answer):
    # GT rows: {"fact_id", "pmid", "score", "join_distance"}
    return [(row["fact_id"], row["score"]) for row in answer]

def baseline_answer_to_scores(answer):
    # molecule W5 baseline rows:
    #   psql/milvus 4-tuple: [fact_id, paper_id, score_dist, join_dist]
    #   dase     5-tuple:     [fact_id, paper_id, fact_text, None, score_dist]
    return [(row[0], row[-1] if len(row) >= 5 else row[2]) for row in answer]

def query_results_to_map(data, is_gt=False):
    fn = gt_answer_to_scores if is_gt else baseline_answer_to_scores
    return {r["query_id"]: fn(r["answer"]) for r in data["results"]}

assert (RESULTS_DIR / GT_FILE).exists(), f"Ground truth not found: {GT_FILE}"
gt_data = load_results(RESULTS_DIR / GT_FILE)
gt_by_qid = query_results_to_map(gt_data, is_gt=True)

BASELINE_FILES = sorted(
    f.name for f in RESULTS_DIR.glob("*.json")
    if f.name != GT_FILE
)

print(f"Ground truth: {GT_FILE}, queries = {len(gt_by_qid)}")
print(f"Baselines ({len(BASELINE_FILES)}):")
for f in BASELINE_FILES:
    print(f"  {f}")

Ground truth: w5_queries_100_gt.json, queries = 100
Baselines (5):
  20260418_073320.json
  20260418_073323.json
  20260418_073327.json
  20260418_073331.json
  20260418_073335.json


In [2]:
def eval_one_baseline(baseline_by_qid, gt_by_qid):
    """recall = |{baseline rows with score <= worst_GT_score}| / K.
    Only queries the baseline actually ran AND that have non-empty GT count."""
    per_query = []
    total_hits = 0
    total_denom = 0
    for qid, answers in baseline_by_qid.items():
        gt_answers = gt_by_qid.get(qid, [])
        n = len(gt_answers)
        if n == 0:
            continue
        an = max(score for _, score in gt_answers)
        m = sum(1 for _, score in answers if score <= an + 1e-6)
        recall = m / n
        per_query.append({"query_id": qid, "K": n, "hits": m, "recall": recall})
        total_hits += m
        total_denom += n
    mean_recall = total_hits / total_denom if total_denom else 0.0
    return per_query, mean_recall

In [3]:
summary = []
per_query_by_baseline = {}
for f in BASELINE_FILES:
    data = load_results(RESULTS_DIR / f)
    by_qid = query_results_to_map(data, is_gt=False)
    per_query, mean_recall = eval_one_baseline(by_qid, gt_by_qid)
    name = Path(f).stem
    n_queries = len(per_query)
    total_sec = data.get("total_elapsed_sec", None)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0
    summary.append({
        "baseline": name,
        "mean_recall": mean_recall,
        "n_queries": len(per_query),
        "qps": qps,
    })
    per_query_by_baseline[name] = per_query
    print(f"{name}: mean_recall = {mean_recall:.4f}, n_queries = {len(per_query)}, qps = {qps:.2f}")

20260418_073320: mean_recall = 0.9945, n_queries = 100, qps = 129.79
20260418_073323: mean_recall = 0.9945, n_queries = 100, qps = 129.46
20260418_073327: mean_recall = 0.9945, n_queries = 100, qps = 127.89
20260418_073331: mean_recall = 0.9945, n_queries = 100, qps = 131.92
20260418_073335: mean_recall = 0.9945, n_queries = 100, qps = 129.80


In [4]:
import pandas as pd

df = pd.DataFrame(summary).sort_values("mean_recall", ascending=False)
print(df.to_string(index=False))

       baseline  mean_recall  n_queries        qps
20260418_073320       0.9945        100 129.791160
20260418_073323       0.9945        100 129.460432
20260418_073327       0.9945        100 127.889551
20260418_073331       0.9945        100 131.923552
20260418_073335       0.9945        100 129.800306
